<a href="https://colab.research.google.com/github/oooinr4018-web/-1/blob/main/__.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 시간 기반 검증 (Temporal Validation)

- 2021~2024년 데이터를 이용하여 예방 우선순위를 산정한 후, 독립된 2025년 데이터를 이용하여 우선순위의 시간적 안정성과 일반화 가능성을 평가

(2021~2024년으로 만든 우선순위가 2025년에도 유지되는지 확인)

- 우리 조에서 도출된 우선순위는 일시적인 결과가 아니라 미래에도 어느 정도 유지되고, '예방' 정책에 적합함을 보여주고자 진행





In [ ]:
# 파일 경로 설정

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# 코랩 왼쪽 Files에 올린 실제 파일명으로 수정
accident_path = '/content/학교안전사고데이터.xlsx'
compensation_path = '/content/학교안전사고보상데이터.xlsx'

In [ ]:
import os

print(os.listdir('/content'))

['.config', '★2021-2025 학교안전사고 데이터.xlsx', '★2021-2025 학교안전사고 보상 데이터.xlsx', 'sample_data']


In [ ]:
# 엑셀 시트 확인

accident_path = '/content/★2021-2025 학교안전사고 데이터.xlsx'
compensation_path = '/content/★2021-2025 학교안전사고 보상 데이터.xlsx'

accident_excel = pd.ExcelFile(accident_path)
compensation_excel = pd.ExcelFile(compensation_path)

print(accident_excel.sheet_names)
print(compensation_excel.sheet_names)

['데이터 설명', '사고유형별 항목', '2021', '2022', '2023', '2024', '2025']
['데이터 설명', '사고유형별 항목', '2021', '2022', '2023', '2024', '2025']


In [ ]:
# 사고 데이터 통합

year_sheets = ['2021', '2022', '2023', '2024', '2025']

accident_list = []

for year in year_sheets:
    temp = pd.read_excel(accident_path, sheet_name=year)

    # 컬럼명 앞뒤 공백 제거
    temp.columns = temp.columns.astype(str).str.strip()

    # 연도 컬럼 추가
    temp['연도'] = int(year)

    accident_list.append(temp)

accident_df = pd.concat(accident_list, ignore_index=True)

print("사고 데이터 크기:", accident_df.shape)
print("\n사고 데이터 컬럼:")
print(accident_df.columns.tolist())

print("\n연도별 사고 건수:")
print(accident_df['연도'].value_counts().sort_index())

display(accident_df.head())

사고 데이터 크기: (865384, 15)

사고 데이터 컬럼:
['구분', '지역', '학교급', '사고자구분', '사고자학년', '사고자성별', '사고연월', '사고발생시각', '사고요일', '사고시간', '사고장소', '사고부위', '사고형태', '사고당시활동', '연도']

연도별 사고 건수:
연도
2021     92920
2022    149017
2023    192724
2024    211148
2025    219575
Name: count, dtype: int64


,구분,지역,학교급,사고자구분,사고자학년,사고자성별,사고연월,사고발생시각,사고요일,사고시간,사고장소,사고부위,사고형태,사고당시활동,연도
0,A0000001,경남,초등학교,일반학생,1학년,여,2014-07,12:40,목,식사시간(간식 포함),특별교실(과학실 외),치아,고정된 물체와의 부딪힘,휴식,2021
1,A0000002,서울,중학교,일반학생,1학년,여,2014-12,11:10,금,체육,운동장,눈,고정된 물체와의 부딪힘,기타 구기,2021
2,A0000003,서울,중학교,일반학생,3학년,남,2015-11,12:8,화,체육,운동장,아래다리(종아리),고정된 물체와의 부딪힘,농구,2021
3,A0000004,경남,초등학교,일반학생,1학년,남,2017-05,13:02,목,쉬는시간,운동장,무릎,고정된 물체와의 부딪힘,기타,2021
4,A0000005,경기,유치원,일반학생,유아,남,2018-07,15:20,월,하교,교통구역(스쿨존 내)-인도,아래다리(종아리),고정된 물체와의 부딪힘,기타,2021


In [ ]:
# 보상 데이터 통합

compensation_list = []

available_comp_sheets = compensation_excel.sheet_names

for year in year_sheets:
    if year in available_comp_sheets:
        temp = pd.read_excel(compensation_path, sheet_name=year)
        temp.columns = temp.columns.astype(str).str.strip()
        temp['연도'] = int(year)
        compensation_list.append(temp)

if len(compensation_list) > 0:
    compensation_df = pd.concat(compensation_list, ignore_index=True)

else:
    # 연도별 시트가 아니라 하나의 시트인 경우
    compensation_df = pd.read_excel(compensation_path)
    compensation_df.columns = compensation_df.columns.astype(str).str.strip()

print("보상 데이터 크기:", compensation_df.shape)
print("\n보상 데이터 컬럼:")
print(compensation_df.columns.tolist())

display(compensation_df.head())

보상 데이터 크기: (528503, 19)

보상 데이터 컬럼:
['구분', '지역', '학교급', '사고자구분', '사고자학년', '사고자성별', '사고시간', '사고장소', '사고부위', '사고형태', '사고당시활동', '요양급여', '장해급여', '간병급여', '유족급여', '장례비', '위로금', '보전비용', '연도']


,구분,지역,학교급,사고자구분,사고자학년,사고자성별,사고시간,사고장소,사고부위,사고형태,사고당시활동,요양급여,장해급여,간병급여,유족급여,장례비,위로금,보전비용,연도
0,F0000001,전남,고등학교,일반학생,2학년,남,쉬는시간,강당(체육관),골반/엉덩이,그밖의 손상 사고,기타,28000,0,0,0,0,0,0,2021
1,F0000002,경남,고등학교,일반학생,1학년,남,그 밖의 교육활동 시간,기타 교외,기타,그밖의 손상 사고,기타,0,0,15027000,0,0,0,0,2021
2,F0000003,광주,초등학교,일반학생,5학년,남,식사시간(간식 포함),일반(교과)교실,복부,그밖의 손상 사고,기타,1968000,0,0,0,0,0,0,2021
3,F0000004,전북,초등학교,일반학생,5학년,남,쉬는시간,일반(교과)교실,어깨,그밖의 손상 사고,기타,1600000,0,0,0,0,0,0,2021
4,F0000005,경북,초등학교,일반학생,1학년,남,체육,운동장,치아,고정된 물체와의 부딪힘,기타,10000,0,0,0,0,0,0,2021


In [ ]:
# 사고유형 변수 확인

group_cols = ['학교급', '사고장소', '사고시간', '사고형태']

missing_cols = [
    col for col in group_cols
    if col not in accident_df.columns
]

if missing_cols:
    raise ValueError(
        f"사고 데이터에 다음 컬럼이 없습니다: {missing_cols}\n"
        f"현재 컬럼: {accident_df.columns.tolist()}"
    )

# 결측치를 '미상'으로 처리
for col in group_cols:
    accident_df[col] = accident_df[col].fillna('미상').astype(str).str.strip()

print("사고유형 생성에 사용할 컬럼:", group_cols)

사고유형 생성에 사용할 컬럼: ['학교급', '사고장소', '사고시간', '사고형태']


In [ ]:
# 사고 데이터와 보상 데이터 연결키 찾기

group_cols = ['학교급', '사고장소', '사고시간', '사고형태']

missing_cols = [
    col for col in group_cols
    if col not in accident_df.columns
]

if missing_cols:
    raise ValueError(
        f"사고 데이터에 다음 컬럼이 없습니다: {missing_cols}\n"
        f"현재 컬럼: {accident_df.columns.tolist()}"
    )

# 결측치를 '미상'으로 처리
for col in group_cols:
    accident_df[col] = accident_df[col].fillna('미상').astype(str).str.strip()

print("사고유형 생성에 사용할 컬럼:", group_cols)

사고유형 생성에 사용할 컬럼: ['학교급', '사고장소', '사고시간', '사고형태']


In [ ]:
amount_cols = [
    '요양급여',
    '장해급여',
    '간병급여',
    '유족급여',
    '장례비',
    '위로금',
    '보전비용'
]

# 금액 컬럼을 숫자로 변환
for col in amount_cols:
    compensation_df[col] = (
        compensation_df[col]
        .astype(str)
        .str.replace(',', '', regex=False)
        .str.replace('원', '', regex=False)
        .str.strip()
    )

    compensation_df[col] = pd.to_numeric(
        compensation_df[col],
        errors='coerce'
    ).fillna(0)

# 총보상액 생성
compensation_df['총보상액'] = compensation_df[amount_cols].sum(axis=1)

compensation_col = '총보상액'

print("총보상액 생성 완료")
print(compensation_df['총보상액'].describe())

display(
    compensation_df[
        ['구분'] + amount_cols + ['총보상액']
    ].head()
)

총보상액 생성 완료
count    5.285030e+05
mean     4.459839e+05
std      5.460694e+06
min      1.000000e+03
25%      5.800000e+04
50%      1.140000e+05
75%      2.730000e+05
max      1.010000e+09
Name: 총보상액, dtype: float64


,구분,요양급여,장해급여,간병급여,유족급여,장례비,위로금,보전비용,총보상액
0,F0000001,28000,0,0,0,0,0,0,28000
1,F0000002,0,0,15027000,0,0,0,0,15027000
2,F0000003,1968000,0,0,0,0,0,0,1968000
3,F0000004,1600000,0,0,0,0,0,0,1600000
4,F0000005,10000,0,0,0,0,0,0,10000


In [ ]:
# 보상 데이터 사고유형 컬럼 정리

# 보상 데이터에도 사고유형 기준 컬럼이 있는지 확인
missing_comp_cols = [
    col for col in group_cols
    if col not in compensation_df.columns
]

if missing_comp_cols:
    raise ValueError(
        f"보상 데이터에 다음 컬럼이 없습니다: {missing_comp_cols}"
    )

# 사고 데이터와 보상 데이터의 문자열 형식 통일
for col in group_cols:
    accident_df[col] = (
        accident_df[col]
        .fillna('미상')
        .astype(str)
        .str.strip()
    )

    compensation_df[col] = (
        compensation_df[col]
        .fillna('미상')
        .astype(str)
        .str.strip()
    )

print("사고유형 기준 컬럼 정리 완료")


사고유형 기준 컬럼 정리 완료


In [ ]:
# 2021-2024와 2025 분리

accident_train = accident_df[
    accident_df['연도'].between(2021, 2024)
].copy()

accident_2025 = accident_df[
    accident_df['연도'] == 2025
].copy()

compensation_train = compensation_df[
    compensation_df['연도'].between(2021, 2024)
].copy()

compensation_2025 = compensation_df[
    compensation_df['연도'] == 2025
].copy()

print("2021~2024 사고 건수:", len(accident_train))
print("2025 사고 건수:", len(accident_2025))

print("2021~2024 보상 건수:", len(compensation_train))
print("2025 보상 건수:", len(compensation_2025))

2021~2024 사고 건수: 645809
2025 사고 건수: 219575
2021~2024 보상 건수: 384608
2025 보상 건수: 143895


In [ ]:
# 2021-2024 사고빈도 X3 생성

frequency_train = (
    accident_train
    .groupby(group_cols, dropna=False)
    .size()
    .reset_index(name='X3_사고빈도')
)

print("2021~2024 사고유형별 빈도 생성 완료")
display(frequency_train.head())

2021~2024 사고유형별 빈도 생성 완료


,학교급,사고장소,사고시간,사고형태,X3_사고빈도
0,고등학교,가정,그 밖의 교육활동 시간,그밖의 손상 사고,1
1,고등학교,가정,그 밖의 교육활동 시간,넘어짐,2
2,고등학교,가정,등교,넘어짐,7
3,고등학교,가정,등교,이동 중 충격을 가함,1
4,고등학교,가정,하교,넘어짐,1


In [ ]:
# 2021-2024 평균 보상액 X2 생성

compensation_summary_train = (
    compensation_train
    .groupby(group_cols, dropna=False)
    .agg(
        X2_평균보상액=('총보상액', 'mean'),
        보상건수=('총보상액', 'size')
    )
    .reset_index()
)

print("2021~2024 사고유형별 평균 보상액 생성 완료")
display(compensation_summary_train.head())

2021~2024 사고유형별 평균 보상액 생성 완료


,학교급,사고장소,사고시간,사고형태,X2_평균보상액,보상건수
0,고등학교,가정,그 밖의 교육활동 시간,그밖의 손상 사고,168000.0,1
1,고등학교,가정,등교,넘어짐,429750.0,4
2,고등학교,가정,등교,이동 중 충격을 가함,418000.0,1
3,고등학교,가정,하교,넘어짐,240000.0,1
4,고등학교,강·바다·하천,그 밖의 교육활동 시간,고정된 물체와의 부딪힘,42000.0,1


In [ ]:
# X2와 X3 결합

priority_train = frequency_train.merge(
    compensation_summary_train,
    on=group_cols,
    how='left'
)

# 보상 기록이 없는 사고유형은 평균보상액 0으로 처리
priority_train['X2_평균보상액'] = (
    priority_train['X2_평균보상액']
    .fillna(0)
)

priority_train['보상건수'] = (
    priority_train['보상건수']
    .fillna(0)
)

priority_train['사고유형'] = (
    priority_train[group_cols]
    .astype(str)
    .agg(' | '.join, axis=1)
)

print("2021~2024 우선순위 데이터 크기:", priority_train.shape)

display(
    priority_train[
        group_cols
        + ['X2_평균보상액', 'X3_사고빈도', '보상건수']
    ].head()
)

2021~2024 우선순위 데이터 크기: (16565, 8)


,학교급,사고장소,사고시간,사고형태,X2_평균보상액,X3_사고빈도,보상건수
0,고등학교,가정,그 밖의 교육활동 시간,그밖의 손상 사고,168000.0,1,1.0
1,고등학교,가정,그 밖의 교육활동 시간,넘어짐,0.0,2,0.0
2,고등학교,가정,등교,넘어짐,429750.0,7,4.0
3,고등학교,가정,등교,이동 중 충격을 가함,418000.0,1,1.0
4,고등학교,가정,하교,넘어짐,240000.0,1,1.0


In [ ]:
# 이상치 영향을 줄이기 위한 로그 변환

priority_train['X2_log'] = np.log1p(
    priority_train['X2_평균보상액']
)

priority_train['X3_log'] = np.log1p(
    priority_train['X3_사고빈도']
)

criteria_cols = ['X2_log', 'X3_log']

display(
    priority_train[
        ['사고유형', 'X2_평균보상액', 'X3_사고빈도', 'X2_log', 'X3_log']
    ].head()
)

,사고유형,X2_평균보상액,X3_사고빈도,X2_log,X3_log
0,고등학교 | 가정 | 그 밖의 교육활동 시간 | 그밖의 손상 사고,168000.0,1,12.031725,0.693147
1,고등학교 | 가정 | 그 밖의 교육활동 시간 | 넘어짐,0.0,2,0.000000,1.098612
2,고등학교 | 가정 | 등교 | 넘어짐,429750.0,7,12.970961,2.079442
3,고등학교 | 가정 | 등교 | 이동 중 충격을 가함,418000.0,1,12.943239,0.693147
4,고등학교 | 가정 | 하교 | 넘어짐,240000.0,1,12.388398,0.693147


In [ ]:
# Min-Max 정규화

def minmax_normalize(series):
    minimum = series.min()
    maximum = series.max()

    if maximum == minimum:
        return pd.Series(
            np.zeros(len(series)),
            index=series.index
        )

    return (series - minimum) / (maximum - minimum)


normalized_train = priority_train.copy()

normalized_cols = []

for col in criteria_cols:
    new_col = f'{col}_norm'

    normalized_train[new_col] = minmax_normalize(
        normalized_train[col]
    )

    normalized_cols.append(new_col)

print("정규화 변수:", normalized_cols)
display(normalized_train[['사고유형'] + normalized_cols].head())

정규화 변수: ['X2_log_norm', 'X3_log_norm']


,사고유형,X2_log_norm,X3_log_norm
0,고등학교 | 가정 | 그 밖의 교육활동 시간 | 그밖의 손상 사고,0.595398,0.000000
1,고등학교 | 가정 | 그 밖의 교육활동 시간 | 넘어짐,0.000000,0.045855
2,고등학교 | 가정 | 등교 | 넘어짐,0.641877,0.156778
3,고등학교 | 가정 | 등교 | 이동 중 충격을 가함,0.640505,0.000000
4,고등학교 | 가정 | 하교 | 넘어짐,0.613048,0.000000


In [ ]:
# CRITIC 가중치 계산

def calculate_critic_weights(df, cols):
    data = df[cols].astype(float)

    std_values = data.std(ddof=0)

    correlation = (
        data.corr()
        .fillna(0)
    )

    conflict = (1 - correlation).sum(axis=1)

    information = std_values * conflict

    if information.sum() == 0:
        weights = pd.Series(
            np.repeat(1 / len(cols), len(cols)),
            index=cols
        )
    else:
        weights = information / information.sum()

    return weights


critic_weights = calculate_critic_weights(
    normalized_train,
    normalized_cols
)

print("CRITIC 가중치")

for criterion, weight in critic_weights.items():
    print(f"{criterion}: {weight:.4f}")

CRITIC 가중치
X2_log_norm: 0.6159
X3_log_norm: 0.3841


In [ ]:
# TOPSIS 점수 계산

def calculate_topsis(df, cols, weights):
    matrix = df[cols].astype(float).copy()

    # 벡터 정규화
    denominator = np.sqrt((matrix ** 2).sum(axis=0))
    denominator = denominator.replace(0, 1)

    normalized_matrix = matrix / denominator

    # CRITIC 가중치 적용
    weighted_matrix = normalized_matrix.mul(
        weights,
        axis=1
    )

    # X2, X3 모두 클수록 위험한 지표
    positive_ideal = weighted_matrix.max(axis=0)
    negative_ideal = weighted_matrix.min(axis=0)

    distance_positive = np.sqrt(
        ((weighted_matrix - positive_ideal) ** 2).sum(axis=1)
    )

    distance_negative = np.sqrt(
        ((weighted_matrix - negative_ideal) ** 2).sum(axis=1)
    )

    score = distance_negative / (
        distance_positive
        + distance_negative
        + 1e-12
    )

    return score


priority_train['TOPSIS_점수'] = calculate_topsis(
    normalized_train,
    normalized_cols,
    critic_weights
)

priority_train = (
    priority_train
    .sort_values('TOPSIS_점수', ascending=False)
    .reset_index(drop=True)
)

priority_train['2021_2024_순위'] = (
    priority_train.index + 1
)

display(
    priority_train[
        [
            '2021_2024_순위',
            '학교급',
            '사고장소',
            '사고시간',
            '사고형태',
            'X2_평균보상액',
            'X3_사고빈도',
            'TOPSIS_점수'
        ]
    ].head(20)
)

,2021_2024_순위,학교급,사고장소,사고시간,사고형태,X2_평균보상액,X3_사고빈도,TOPSIS_점수
0,1,중학교,운동장,체육,넘어짐,4.987837e+05,13327,0.843288
1,2,고등학교,운동장,체육,넘어짐,1.223242e+06,6451,0.837645
2,3,중학교,강당(체육관),체육,넘어짐,4.485044e+05,9749,0.834275
3,4,중학교,운동장,체육,고정된 물체와의 부딪힘,2.948598e+05,11303,0.829817
4,5,초등학교,강당(체육관),체육,넘어짐,2.565370e+05,12631,0.829345
5,6,고등학교,강당(체육관),체육,넘어짐,7.176397e+05,6592,0.828520
6,7,중학교,강당(체육관),체육,고정된 물체와의 부딪힘,2.219905e+05,13842,0.828027
7,8,중학교,강당(체육관),체육,스포츠 활동 중 충격을 가함,2.818045e+05,8660,0.821426
8,9,중학교,강당(체육관),체육,움직이는 물체와의 부딪힘,1.698085e+05,12615,0.821352
9,10,고등학교,강당(체육관),체육,스포츠 활동 중 충격을 가함,4.724573e+05,6659,0.821013


In [ ]:
# 2025년 사고빈도와 보상액 생성

frequency_2025 = (
    accident_2025
    .groupby(group_cols, dropna=False)
    .size()
    .reset_index(name='사고빈도_2025')
)

compensation_summary_2025 = (
    compensation_2025
    .groupby(group_cols, dropna=False)
    .agg(
        평균보상액_2025=('총보상액', 'mean'),
        보상건수_2025=('총보상액', 'size')
    )
    .reset_index()
)

priority_2025 = frequency_2025.merge(
    compensation_summary_2025,
    on=group_cols,
    how='left'
)

priority_2025['평균보상액_2025'] = (
    priority_2025['평균보상액_2025']
    .fillna(0)
)

priority_2025['보상건수_2025'] = (
    priority_2025['보상건수_2025']
    .fillna(0)
)

priority_2025['사고유형'] = (
    priority_2025[group_cols]
    .astype(str)
    .agg(' | '.join, axis=1)
)

priority_2025['X2_log'] = np.log1p(
    priority_2025['평균보상액_2025']
)

priority_2025['X3_log'] = np.log1p(
    priority_2025['사고빈도_2025']
)

print("2025 사고유형 수:", len(priority_2025))
display(priority_2025.head())

2025 사고유형 수: 10639


,학교급,사고장소,사고시간,사고형태,사고빈도_2025,평균보상액_2025,보상건수_2025,사고유형,X2_log,X3_log
0,고등학교,가정,등교,넘어짐,2,144500.0,2.0,고등학교 | 가정 | 등교 | 넘어짐,11.881042,1.098612
1,고등학교,가정,등교,움직이는 물체와의 부딪힘,1,220000.0,1.0,고등학교 | 가정 | 등교 | 움직이는 물체와의 부딪힘,12.301387,0.693147
2,고등학교,가정,하교,넘어짐,1,2658000.0,1.0,고등학교 | 가정 | 하교 | 넘어짐,14.793085,0.693147
3,고등학교,강·바다·하천,그 밖의 교육활동 시간,고정된 물체와의 부딪힘,1,0.0,0.0,고등학교 | 강·바다·하천 | 그 밖의 교육활동 시간 | 고정된 물체와의 부딪힘,0.000000,0.693147
4,고등학교,강·바다·하천,그 밖의 교육활동 시간,이동 중 충격을 가함,1,1088000.0,1.0,고등학교 | 강·바다·하천 | 그 밖의 교육활동 시간 | 이동 중 충격을 가함,13.899853,0.693147


In [ ]:
# 2025년 TOPSIS 실제 순위 계산

# 2025 데이터 정규화
normalized_2025 = priority_2025.copy()

for col in criteria_cols:
    normalized_2025[f'{col}_norm'] = minmax_normalize(
        normalized_2025[col]
    )

# 2021~2024에서 계산한 CRITIC 가중치를 그대로 적용
priority_2025['TOPSIS_점수_2025'] = calculate_topsis(
    normalized_2025,
    normalized_cols,
    critic_weights
)

priority_2025 = (
    priority_2025
    .sort_values('TOPSIS_점수_2025', ascending=False)
    .reset_index(drop=True)
)

priority_2025['실제순위_2025'] = priority_2025.index + 1

display(
    priority_2025[
        [
            '실제순위_2025',
            '학교급',
            '사고장소',
            '사고시간',
            '사고형태',
            '평균보상액_2025',
            '사고빈도_2025',
            'TOPSIS_점수_2025'
        ]
    ].head(20)
)

,실제순위_2025,학교급,사고장소,사고시간,사고형태,평균보상액_2025,사고빈도_2025,TOPSIS_점수_2025
0,1,중학교,강당(체육관),체육,스포츠 활동 중 충격을 가함,335561.866749,5207,0.834857
1,2,중학교,강당(체육관),체육,움직이는 물체와의 부딪힘,191344.331267,7737,0.834470
2,3,초등학교,강당(체육관),체육,스포츠 활동 중 충격을 가함,234822.337662,5937,0.832460
3,4,고등학교,강당(체육관),체육,스포츠 활동 중 충격을 가함,552830.523130,3549,0.826198
4,5,중학교,운동장,체육,움직이는 물체와의 부딪힘,205251.085142,5083,0.824942
5,6,중학교,운동장,체육,스포츠 활동 중 충격을 가함,355447.761194,3593,0.819131
6,7,초등학교,강당(체육관),체육,움직이는 물체와의 부딪힘,149569.074334,4892,0.817776
7,8,고등학교,강당(체육관),체육,움직이는 물체와의 부딪힘,262555.684455,2938,0.802318
8,9,초등학교,강당(체육관),체육,넘어짐,244820.965231,2920,0.800770
9,10,중학교,운동장,체육,넘어짐,559537.304452,2364,0.800182


In [ ]:
# TOP20 일치율 검증

predicted_top20 = set(
    priority_train
    .head(20)['사고유형']
)

actual_top20 = set(
    priority_2025
    .head(20)['사고유형']
)

common_top20 = predicted_top20 & actual_top20

top20_overlap_count = len(common_top20)
top20_overlap_rate = top20_overlap_count / 20

union_count = len(predicted_top20 | actual_top20)

if union_count == 0:
    jaccard_score = 0
else:
    jaccard_score = len(common_top20) / union_count

print("공통 TOP20 사고유형 수:", top20_overlap_count)
print(f"TOP20 일치율: {top20_overlap_rate:.2%}")
print(f"Jaccard 유사도: {jaccard_score:.4f}")

print("\n공통 사고유형:")

for accident_type in sorted(common_top20):
    print("-", accident_type)

공통 TOP20 사고유형 수: 14
TOP20 일치율: 70.00%
Jaccard 유사도: 0.5385

공통 사고유형:
- 고등학교 | 강당(체육관) | 체육 | 넘어짐
- 고등학교 | 강당(체육관) | 체육 | 스포츠 활동 중 충격을 가함
- 고등학교 | 강당(체육관) | 체육 | 움직이는 물체와의 부딪힘
- 고등학교 | 운동장 | 체육 | 스포츠 활동 중 충격을 가함
- 중학교 | 강당(체육관) | 체육 | 넘어짐
- 중학교 | 강당(체육관) | 체육 | 스포츠 활동 중 충격을 가함
- 중학교 | 강당(체육관) | 체육 | 움직이는 물체와의 부딪힘
- 중학교 | 운동장 | 체육 | 넘어짐
- 중학교 | 운동장 | 체육 | 스포츠 활동 중 충격을 가함
- 중학교 | 운동장 | 체육 | 움직이는 물체와의 부딪힘
- 초등학교 | 강당(체육관) | 체육 | 고정된 물체와의 부딪힘
- 초등학교 | 강당(체육관) | 체육 | 넘어짐
- 초등학교 | 강당(체육관) | 체육 | 스포츠 활동 중 충격을 가함
- 초등학교 | 강당(체육관) | 체육 | 움직이는 물체와의 부딪힘


- 2021~2024년 데이터를 기반으로 선정한 우선관리 사고유형 중 70%가 2025년에도 Top20에 포함됨



In [ ]:
# SPearsman 순위상관 검증

from scipy.stats import spearmanr

rank_comparison = priority_train[
    [
        '사고유형',
        '2021_2024_순위',
        'TOPSIS_점수'
    ]
].merge(
    priority_2025[
        [
            '사고유형',
            '실제순위_2025',
            'TOPSIS_점수_2025'
        ]
    ],
    on='사고유형',
    how='inner'
)

print("두 기간에 공통으로 존재한 사고유형 수:", len(rank_comparison))

if len(rank_comparison) >= 2:
    spearman_corr, spearman_pvalue = spearmanr(
        rank_comparison['2021_2024_순위'],
        rank_comparison['실제순위_2025']
    )

    print(f"Spearman 순위상관계수: {spearman_corr:.4f}")
    print(f"p-value: {spearman_pvalue:.6f}")

else:
    print("공통 사고유형이 2개 미만이라 Spearman 상관계수를 계산할 수 없습니다.")

display(
    rank_comparison
    .sort_values('2021_2024_순위')
    .head(20)
)

두 기간에 공통으로 존재한 사고유형 수: 8279
Spearman 순위상관계수: 0.7418
p-value: 0.000000


,사고유형,2021_2024_순위,TOPSIS_점수,실제순위_2025,TOPSIS_점수_2025
0,중학교 | 운동장 | 체육 | 넘어짐,1,0.843288,10,0.800182
1,고등학교 | 운동장 | 체육 | 넘어짐,2,0.837645,22,0.737140
2,중학교 | 강당(체육관) | 체육 | 넘어짐,3,0.834275,12,0.782414
3,중학교 | 운동장 | 체육 | 고정된 물체와의 부딪힘,4,0.829817,70,0.659805
4,초등학교 | 강당(체육관) | 체육 | 넘어짐,5,0.829345,9,0.800770
5,고등학교 | 강당(체육관) | 체육 | 넘어짐,6,0.828520,20,0.744744
6,중학교 | 강당(체육관) | 체육 | 고정된 물체와의 부딪힘,7,0.828027,38,0.705699
7,중학교 | 강당(체육관) | 체육 | 스포츠 활동 중 충격을 가함,8,0.821426,1,0.834857
8,중학교 | 강당(체육관) | 체육 | 움직이는 물체와의 부딪힘,9,0.821352,2,0.834470
9,고등학교 | 강당(체육관) | 체육 | 스포츠 활동 중 충격을 가함,10,0.821013,4,0.826198


- 보통 0.7 이상이면 높은 일관성, 0.5~0.7이면 보통, 0.3 이하면 낮음.

- 결괏값이 0.7418 (높은 순위 일관성)

- 2021~2024년에 위험하다고 평가된 사고유형은 2025년에도 대체로 높은 위험순위를 유지함.

In [ ]:
# 2021-2024 예측 TOP20이 2025년에도 위험했는지 비교

validation_df = priority_2025.copy()

validation_df['예측_TOP20_여부'] = (
    validation_df['사고유형']
    .isin(predicted_top20)
)

validation_summary = (
    validation_df
    .groupby('예측_TOP20_여부')
    .agg(
        사고유형수=('사고유형', 'count'),
        평균_사고빈도_2025=('사고빈도_2025', 'mean'),
        중앙값_사고빈도_2025=('사고빈도_2025', 'median'),
        평균_보상액_2025=('평균보상액_2025', 'mean'),
        평균_TOPSIS_2025=('TOPSIS_점수_2025', 'mean')
    )
)

validation_summary = validation_summary.rename(
    index={
        False: '예측 TOP20 이외',
        True: '2021~2024 예측 TOP20'
    }
)

display(validation_summary)

,사고유형수,평균_사고빈도_2025,중앙값_사고빈도_2025,평균_보상액_2025,평균_TOPSIS_2025
예측_TOP20_여부,,,,,
예측 TOP20 이외,10619,15.532065,2.0,569197.746584,0.228040
2021~2024 예측 TOP20,20,2732.000000,2165.0,469008.036080,0.767976


- 예측 TOP20 평균 사고빈도가 2732건, 나머지 평균 사고빈도가 15건

- 2021-2024년에 위험하다고 선정한 유형들이 2025년에도 실제 사고 훨씬 많이 발생 (2732건)



In [ ]:
# 예측 TOP20 상세 비교표

top20_comparison = priority_train[
    [
        '사고유형',
        '2021_2024_순위',
        'X2_평균보상액',
        'X3_사고빈도',
        'TOPSIS_점수'
    ]
].head(20).merge(
    priority_2025[
        [
            '사고유형',
            '실제순위_2025',
            '평균보상액_2025',
            '사고빈도_2025',
            'TOPSIS_점수_2025'
        ]
    ],
    on='사고유형',
    how='left'
)

top20_comparison['순위차이'] = (
    top20_comparison['실제순위_2025']
    - top20_comparison['2021_2024_순위']
)

display(top20_comparison)

,사고유형,2021_2024_순위,X2_평균보상액,X3_사고빈도,TOPSIS_점수,실제순위_2025,평균보상액_2025,사고빈도_2025,TOPSIS_점수_2025,순위차이
0,중학교 | 운동장 | 체육 | 넘어짐,1,4.987837e+05,13327,0.843288,10,5.595373e+05,2364,0.800182,9
1,고등학교 | 운동장 | 체육 | 넘어짐,2,1.223242e+06,6451,0.837645,22,1.295898e+06,971,0.737140,20
2,중학교 | 강당(체육관) | 체육 | 넘어짐,3,4.485044e+05,9749,0.834275,12,4.512192e+05,1945,0.782414,9
3,중학교 | 운동장 | 체육 | 고정된 물체와의 부딪힘,4,2.948598e+05,11303,0.829817,70,3.729819e+05,492,0.659805,66
4,초등학교 | 강당(체육관) | 체육 | 넘어짐,5,2.565370e+05,12631,0.829345,9,2.448210e+05,2920,0.800770,4
5,고등학교 | 강당(체육관) | 체육 | 넘어짐,6,7.176397e+05,6592,0.828520,20,8.321465e+05,1120,0.744744,14
6,중학교 | 강당(체육관) | 체육 | 고정된 물체와의 부딪힘,7,2.219905e+05,13842,0.828027,38,3.302867e+05,822,0.705699,31
7,중학교 | 강당(체육관) | 체육 | 스포츠 활동 중 충격을 가함,8,2.818045e+05,8660,0.821426,1,3.355619e+05,5207,0.834857,-7
8,중학교 | 강당(체육관) | 체육 | 움직이는 물체와의 부딪힘,9,1.698085e+05,12615,0.821352,2,1.913443e+05,7737,0.834470,-7
9,고등학교 | 강당(체육관) | 체육 | 스포츠 활동 중 충격을 가함,10,4.724573e+05,6659,0.821013,4,5.528305e+05,3549,0.826198,-6


- X2,X3 두 지표만으로도 2025년 데이터에서 Top20 일치율 70%, Spearman 0.7418을 보여 예방 우선순위를 안정적으로 산정 가능. X1을 추가변수로 도입하는 것은 향후 확장 방향으로 고려

- 원자료에서 즉시 계산 가능한 X2, X3을 이용하여 CRITIC+TOPSIS 과정 진행.

- X1은 원자료에 존재하는 변수가 아니라, 분류모델이 산출하는 예측확률. X1을 생성하기위해서는 실제 고위험 여부 또는 위험등급과 같은 타깃 변수가 필요함.

-  X2와 X3을 이용하여 임의로 위험등급을 만들어 그 위험등급을 예측하면 X1 생성 가능. 하지만 X1 안에 X2,X3의 정보가 포함되어 중복 반영될 수 있는 가능성 존재.

- 현재 분석에서는 의미가 명확한 X2,X3만을 사용해 CRITIC-TOPSIS 수행하는 것이 더 타당하다고 생각.

